In [1]:
import pandas as pd

In [7]:
data1 = pd.read_csv(r"C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\uniprotkb_ec_3_4_21_AND_length_450_TO_5_2026_04_14.tsv\uniprotkb_ec_3_4_21_AND_length_450_TO_5_2026_04_14.tsv", sep = '\t')

In [7]:
asp = pd.read_csv(r"C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\Aspatic Protease.txt", sep = '\t')

In [8]:
asp

,MEROPS ID,Peptidase,Total substrates,Physiological substrates,Non-physiological substrates
0,A01.001,pepsin A,970,21,949
1,A01.002,pepsin B,25,1,24
2,A01.003,gastricsin,50,6,44
3,A01.004,BACE1 g.p. (Homo sapiens),25,16,9
4,A01.006,chymosin,20,11,9
...,...,...,...,...,...
297,A9G.010,pycnoporopepsin,8,0,8
298,A9G.017,acid peptidase (Cladosporium),8,0,8
299,A9G.018,acid peptidase (Paecilomyces),0,0,0
300,A9G.019,acrocylindropepsin,6,0,6


In [10]:
mer = asp['MEROPS ID']

In [12]:
import requests
import pandas as pd
import time
import re
from bs4 import BeautifulSoup


# ============================================================
# INPUT
# ============================================================

# Your existing list:
#
# mer = [
#     "S01.001",
#     "S01.002",
#     "M01.001",
#     ...
# ]


# ============================================================
# SETTINGS
# ============================================================

MEROPS_URL = "https://www.ebi.ac.uk/merops/cgi-bin/substrates"
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/{}.fasta"

WINDOW = 10
MEROPS_DELAY = 1.0


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 Protease-substrate research"
})


# ============================================================
# CACHE
# ============================================================

merops_cache = {}
sequence_cache = {}


# ============================================================
# GET MEROPS SUBSTRATE TABLE
# ============================================================

def get_merops_substrates(merops_id):

    if merops_id in merops_cache:
        return merops_cache[merops_id]

    try:

        response = session.get(
            MEROPS_URL,
            params={"id": merops_id},
            timeout=60
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        tables = soup.find_all("table")

        for table in tables:

            rows = table.find_all("tr")

            if not rows:
                continue

            headers = [
                cell.get_text(" ", strip=True)
                for cell in rows[0].find_all(
                    ["th", "td"]
                )
            ]

            # Find the substrate table
            substrate_column = None

            for i, header in enumerate(headers):

                if "substrate" in header.lower():

                    substrate_column = i
                    break

            if substrate_column is None:
                continue

            data = []

            for row in rows[1:]:

                cells = [
                    cell.get_text(" ", strip=True)
                    for cell in row.find_all(
                        ["td", "th"]
                    )
                ]

                if len(cells) <= substrate_column:
                    continue

                # Keep the complete row
                if len(cells) < len(headers):
                    cells += [""] * (
                        len(headers) - len(cells)
                    )

                cells = cells[:len(headers)]

                row_dict = dict(
                    zip(headers, cells)
                )

                data.append(row_dict)

            merops_cache[merops_id] = data

            return data

        merops_cache[merops_id] = []

        return []

    except Exception as e:

        print(
            f"MEROPS ERROR {merops_id}: {e}"
        )

        merops_cache[merops_id] = []

        return []


# ============================================================
# GENERIC COLUMN FINDER
# ============================================================

def get_column(row, possible_names):

    normalized = {}

    for key in row.keys():

        clean = re.sub(
            r"[^a-z0-9]",
            "",
            str(key).lower()
        )

        normalized[clean] = key

    for name in possible_names:

        clean = re.sub(
            r"[^a-z0-9]",
            "",
            name.lower()
        )

        if clean in normalized:

            value = row[normalized[clean]]

            if value is not None:
                return str(value).strip()

    return ""


# ============================================================
# GET SUBSTRATE UniProt ID
# ============================================================

def get_substrate_uniprot(row):

    value = get_column(
        row,
        [
            "UniProt",
            "Uniprot",
            "UniProt ID",
            "UniProt accession",
            "Accession"
        ]
    )

    # Some MEROPS entries may contain additional text.
    # Try to extract a standard UniProt accession.

    match = re.search(
        r"\b[A-NR-Z][0-9][A-Z0-9]{3}[0-9]\b",
        value
    )

    if match:
        return match.group(0)

    return value


# ============================================================
# GET SUBSTRATE SEQUENCE FROM UniProt
# ============================================================

def get_uniprot_sequence(uniprot_id):

    if not uniprot_id:
        return None

    if uniprot_id in sequence_cache:
        return sequence_cache[uniprot_id]

    try:

        # Remove isoform suffix if present
        # e.g. P12345-2 -> P12345
        base_id = uniprot_id.split("-")[0]

        url = UNIPROT_URL.format(
            base_id
        )

        response = session.get(
            url,
            timeout=60
        )

        if response.status_code != 200:

            sequence_cache[uniprot_id] = None

            return None

        sequence = "".join(
            line.strip()
            for line in response.text.splitlines()
            if not line.startswith(">")
        )

        sequence_cache[
            uniprot_id
        ] = sequence

        return sequence

    except Exception as e:

        print(
            f"UniProt ERROR {uniprot_id}: {e}"
        )

        sequence_cache[
            uniprot_id
        ] = None

        return None


# ============================================================
# PARSE CLEAVAGE POSITION
# ============================================================

def parse_cleavage_position(cleavage_site):

    if not cleavage_site:
        return None

    # Examples:
    #
    # Gly373+Ala
    # Arg195+Ser
    # Lys1754+Ser
    #
    # The number corresponds to P1.

    match = re.search(
        r"[A-Za-z]{1,3}[- ]?(\d+)\s*\+",
        cleavage_site
    )

    if match:

        return int(
            match.group(1)
        )

    return None


# ============================================================
# EXTRACT ±10 AA WINDOW
# ============================================================

def get_binding_window(
    sequence,
    cleavage_position,
    window=10
):

    if (
        not sequence
        or cleavage_position is None
    ):
        return None

    # MEROPS uses 1-based residue numbering.
    #
    # If cleavage is:
    #
    # residue 100 | residue 101
    #       P1    |    P1'
    #
    # Python cleavage boundary = index 100

    cut_index = cleavage_position

    left = max(
        0,
        cut_index - window
    )

    right = min(
        len(sequence),
        cut_index + window
    )

    region = sequence[
        left:right
    ]

    marker = cut_index - left

    return (
        region[:marker]
        + "|"
        + region[marker:]
    )


# ============================================================
# MAIN LOOP
# ============================================================

results = []

failed_merops = []

for i, merops_id in enumerate(
    mer,
    start=1
):

    print(
        f"[{i}/{len(mer)}] {merops_id}"
    )

    # --------------------------------------------------------
    # Get substrate records
    # --------------------------------------------------------

    substrates = get_merops_substrates(
        merops_id
    )

    print(
        f"    {len(substrates)} substrate records"
    )

    if not substrates:
        failed_merops.append(
            merops_id
        )

    # --------------------------------------------------------
    # Process every substrate
    # --------------------------------------------------------

    for substrate in substrates:

        # ----------------------------------------------------
        # Substrate information
        # ----------------------------------------------------

        substrate_name = get_column(
            substrate,
            [
                "Substrate",
                "Substrate name"
            ]
        )

        substrate_uniprot = (
            get_substrate_uniprot(
                substrate
            )
        )

        # ----------------------------------------------------
        # Cleavage site
        # ----------------------------------------------------

        cleavage_site = get_column(
            substrate,
            [
                "Cleavage Site",
                "Cleavage",
                "Site"
            ]
        )

        # ----------------------------------------------------
        # Get full substrate sequence
        # ----------------------------------------------------

        substrate_sequence = None

        if substrate_uniprot:

            substrate_sequence = (
                get_uniprot_sequence(
                    substrate_uniprot
                )
            )

        # ----------------------------------------------------
        # Synthetic substrate
        #
        # If there is no UniProt sequence, look for a
        # sequence directly in the MEROPS record.
        # ----------------------------------------------------

        if not substrate_sequence:

            substrate_sequence = get_column(
                substrate,
                [
                    "Sequence",
                    "Substrate sequence",
                    "Peptide sequence"
                ]
            )

            if not substrate_sequence:
                substrate_sequence = None

        # ----------------------------------------------------
        # Cleavage position
        # ----------------------------------------------------

        cleavage_position = (
            parse_cleavage_position(
                cleavage_site
            )
        )

        # ----------------------------------------------------
        # ±10 AA sequence
        # ----------------------------------------------------

        binding_window = (
            get_binding_window(
                substrate_sequence,
                cleavage_position,
                WINDOW
            )
        )

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        results.append({

            "merops_id":
                merops_id,

            "substrate":
                substrate_name,

            "substrate_uniprot":
                substrate_uniprot,

            "substrate_sequence":
                substrate_sequence,

            "cleavage_site":
                cleavage_site,

            "cleavage_position":
                cleavage_position,

            "binding_window_±10":
                binding_window
        })

    # --------------------------------------------------------
    # Delay before next MEROPS request
    # --------------------------------------------------------

    time.sleep(
        MEROPS_DELAY
    )


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    results
)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "merops_substrates_with_window10.csv",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINISHED")
print("=" * 70)

print(
    "MEROPS IDs:",
    len(mer)
)

print(
    "Total substrate records:",
    len(df)
)

print(
    "Records with substrate sequence:",
    df["substrate_sequence"].notna().sum()
)

print(
    "Records with ±10 AA window:",
    df["binding_window_±10"].notna().sum()
)

print(
    "MEROPS IDs with no substrate records:",
    len(failed_merops)
)

print(
    "\nCSV:",
    "merops_substrates_with_window10.csv"
)

display(
    df.head(20)
)

[1/302] A01.001
    970 substrate records
[2/302] A01.002
    25 substrate records
[3/302] A01.003
    50 substrate records
[4/302] A01.004
    25 substrate records
[5/302] A01.006
    20 substrate records
[6/302] A01.007
    10 substrate records
[7/302] A01.008
    3 substrate records
[8/302] A01.009
    924 substrate records
[9/302] A01.010
    1596 substrate records
UniProt ERROR Q99LD4: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
[10/302] A01.011
    17 substrate records
[11/302] A01.012
    249 substrate records
[12/302] A01.013
    9 substrate records
[13/302] A01.014
    26 substrate records
[14/302] A01.015
    1 substrate records
[15/302] A01.016
    157 substrate records
[16/302] A01.017
    11 substrate records
[17/302] A01.018
    14 substrate records
[18/302] A01.019
    11 substrate records
[19/302] A01.020
    62 substrate records
[20/302] A01.021
    5 substrate records
[21/302]

,merops_id,substrate,substrate_uniprot,substrate_sequence,cleavage_site,cleavage_position,binding_window_±10
0,A01.001,ac-Phe-Tyr(I2),,None,ac-Phe+Tyr(I2),NaN,None
1,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Asn7+Lys-peptide,7.0,MSFTLTN|KNVIFVAGLG
2,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Asn9+Val-peptide,9.0,MSFTLTNKN|VIFVAGLGGI
3,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Phe12+Val-peptide,12.0,FTLTNKNVIF|VAGLGGIGLD
4,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Gly18+Ile-peptide,18.0,NVIFVAGLGG|IGLDTSKELL
5,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Gly20+Leu-peptide,20.0,IFVAGLGGIG|LDTSKELLKR
6,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Leu21+Asp-peptide,21.0,FVAGLGGIGL|DTSKELLKRD
7,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Leu28+Lys-peptide,28.0,IGLDTSKELL|KRDLKNLVIL
8,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Leu27+Leu-peptide,27.0,GIGLDTSKEL|LKRDLKNLVI
9,A01.001,alcohol dehydrogenase,P00334,MSFTLTNKNVIFVAGLGGIGLDTSKELLKRDLKNLVILDRIENPAA...,peptide-Arg30+Asp-peptide,30.0,LDTSKELLKR|DLKNLVILDR


In [14]:
df['binding_window_±10'].nunique()

4543

In [16]:
df.to_csv('Aspatic Protease Substrates.csv')

In [17]:
data1

,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Mass,Sequence,Active site,Binding site,Protein existence,Protein families
0,A0A096N5A4,unreviewed,A0A096N5A4_PAPAN,"Serine protease HTRA2, mitochondrial (EC 3.4.2...",HTRA2,Papio anubis (Olive baboon),458,48746,MAAPRAGRGAGWSLRAWRALGGIRWGRRPRLTPDLRALLTSGTSDP...,NaN,NaN,Inferred from homology,Peptidase S1C family
1,A0A0D9RT82,unreviewed,A0A0D9RT82_CHLSB,"Serine protease HTRA2, mitochondrial (EC 3.4.2...",HTRA2,Chlorocebus sabaeus (Green monkey) (Simia sabaea),458,48755,MAAPRAGRGAGWSLRAWRALGGIRWGRRPRLTPDLRALLTSGTSDP...,NaN,NaN,Inferred from homology,Peptidase S1C family
2,A0A0P6IWL7,unreviewed,A0A0P6IWL7_HETGA,Plasminogen activator (EC 3.4.21.68),PLAT Plat,Heterocephalus glaber (Naked mole rat),562,63557,MRMRQELLCVLLLCGAISTLPSQKTHTRFRRGARSYRVTCRDEKTQ...,"ACT_SITE 357; /note=""Charge relay system""; /ev...",NaN,Inferred from homology,Peptidase S1 family
3,A0A1U7QSP2,unreviewed,A0A1U7QSP2_MESAU,"Serine protease HTRA2, mitochondrial (EC 3.4.2...",Htra2,Mesocricetus auratus (Golden hamster),458,48832,MAALKAGRGAGWSLRAWRALGGIFWGKGPLLTPDLRGLLTSGSPDP...,NaN,NaN,Inferred from homology,Peptidase S1C family
4,A0A286XQ11,unreviewed,A0A286XQ11_CAVPO,"Serine protease HTRA2, mitochondrial (EC 3.4.2...",HTRA2,Cavia porcellus (Guinea pig),455,48645,MAALRAGRGAAWILQGWRALGGILCGKGPLLTPDLRALLTSGTPYL...,NaN,NaN,Inferred from homology,Peptidase S1C family
...,...,...,...,...,...,...,...,...,...,...,...,...,...
42642,X0HJK8,unreviewed,X0HJK8_FUSOX,Serine protease (EC 3.4.21.-),FOPG_08729,Fusarium oxysporum f. sp. conglutinans race 2 ...,577,62150,MMGTGWLIRPDLLVTAGHVVHDWGRKLGPADQIKCYIGYNGKQSVG...,NaN,NaN,Inferred from homology,Peptidase S1B family
42643,X0IUL2,unreviewed,X0IUL2_FUSOX,Serine protease (EC 3.4.21.-),FOPG_08591,Fusarium oxysporum f. sp. conglutinans race 2 ...,589,63911,MASSNPERTYARWSLRDVPHTLATDVLPIGPGADGIETVFDDDSRR...,NaN,NaN,Inferred from homology,Peptidase S1 family; Peptidase S1B family
42644,X0L973,unreviewed,X0L973_FUSOX,Serine protease (EC 3.4.21.-),FOTG_14251,Fusarium oxysporum f. sp. vasinfectum 25433,577,62087,MMGTGWLIRPDLLVTAGHVVHDWGRKLGPADQIKCYIGYNGKQSVG...,NaN,NaN,Inferred from homology,Peptidase S1B family
42645,X5DRZ5,unreviewed,X5DRZ5_9CORY,Putative trypsin-like serine protease (EC 3.4....,pepD CGLY_04870,Corynebacterium glyciniphilum AJ 3170,475,47351,MSGEDRIGNGDDGFSDPRDPRDPRGEQGHQGPGVWGRDGRGSSSPE...,NaN,NaN,Inferred from homology,Peptidase S1C family


In [16]:
data1['Protein families'].nunique()

109

In [31]:
prot = list(data1['Entry'].unique())

In [32]:
len(prot)

42647

In [2]:
path = r"C:\Users\apkar\Downloads\uniprotkb_serine_merops_substrates_with_sequences.csv"
data = pd.read_csv(path)

In [4]:
data['substrate_sequence'].nunique()

101

In [5]:
data['input_uniprot'].nunique()

822

In [26]:
import numpy as np

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
#----------------------------------------------------------------------------------------------------------------------
# Add the path for the protease dataframe 

# data1 = pd.read_csv(r"uniprotkb_ec_3_4_21_AND_length_450_TO_5_2026_04_14.tsv", sep = '\t')
# prot = list(data1['Entry'].unique())

#----------------------------------------------------------------------------------------------------------------------
def get_merops_id(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.txt"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code != 200:
            return uniprot_id, None

        merops_ids = []

        for line in response.text.splitlines():
            if line.startswith("DR   MEROPS;"):
                parts = line.split(";")
                if len(parts) > 1:
                    merops_ids.append(parts[1].strip().rstrip("."))

        return uniprot_id, "; ".join(merops_ids) if merops_ids else None

    except Exception:
        return uniprot_id, None


# Your UniProt IDs
uniprot_ids = prot

results = []

# Number of simultaneous requests
MAX_WORKERS = 10

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(get_merops_id, uid)
        for uid in uniprot_ids
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Getting MEROPS IDs"
    ):
        results.append(future.result())


# Convert results to DataFrame
merops_df = pd.DataFrame(
    results,
    columns=["UniProt_ID", "MEROPS_ID"]
)

# Add MEROPS IDs back to your original dataframe



Getting MEROPS IDs: 100%|██████████| 42647/42647 [6:58:40<00:00,  1.70it/s]   


In [36]:
l = len(merops_df)
l2 = len(merops_df.dropna())
print('Numeber of Proteases with MEROPS ID : ' ,l2)
print('Percentage of Proteases with MEROPS ID : ' , l2 * 100/l, '%')

Numeber of Proteases with MEROPS ID :  1220
Percentage of Proteases with MEROPS ID :  2.860693600956691 %


In [37]:
filtered = merops_df.dropna()

In [41]:
filtered.to_csv(r'C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\Protease_merops.csv')

In [4]:
merops = pd.read_csv(r'C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\Protease_merops.csv')

In [2]:
ser = pd.read_csv(r"C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\hip1_similarity_final compiled.csv")

In [5]:
merops

,Unnamed: 0,UniProt_ID,MEROPS_ID
0,57,A0JNK3,S01.278
1,60,A6YFB5,S01.277
2,61,F6UNS8,S01.278
3,63,B0BNB9,S01.278
4,64,G1PU69,S01.278
...,...,...,...
1215,42622,V9W7L2,S55.001
1216,42624,W5SHP2,S01.525
1217,42628,W5SMW8,S41.013
1218,42632,W5SZQ1,S41.013


In [3]:
ser

,rank,accession,length,C,best_band,AAAA,AAAD,AADA,AADD,ADAA,...,ADDA,ADDD,DAAA,DAAD,DADA,DADD,DDAA,DDAD,DDDA,DDDD
0,1,P65824,520,1.000000,AAAD,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,2,P9WHR3,520,1.000000,AAAD,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
2,3,A0A097R7X2,525,0.742027,AAAA,0.742027,0.308359,0.396374,0.371211,0.437498,...,0.336528,0.331605,0.316790,0.410445,0.359452,0.327044,0.442840,0.400604,0.274666,0.495306
3,4,D5RI84,475,0.710195,AAAA,0.710195,0.433392,0.634988,0.307869,0.223498,...,0.347195,0.349190,0.303696,0.310790,0.346881,0.321472,0.308044,0.290831,0.448359,0.388990
4,5,Q28278,456,0.705648,DAAD,0.285025,0.390098,0.451574,0.400482,0.286399,...,0.466489,0.304141,0.378106,0.705648,0.381739,0.278645,0.426911,0.451948,0.440053,0.362939
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1214,1215,Q89504,568,0.412353,DDDA,0.381894,0.346296,0.273213,0.363967,0.285818,...,0.267852,0.319979,0.385939,0.331831,0.242935,0.277175,0.388185,0.322473,0.412353,0.373821
1215,1216,E7BAR3,581,0.408235,AAAA,0.408235,0.383973,0.321834,0.322990,0.378860,...,0.280850,0.378739,0.380688,0.301842,0.338597,0.252217,0.387749,0.379218,0.281888,0.384788
1216,1217,A6TEP2,455,0.406265,DDDA,0.345258,0.266788,0.384372,0.385949,0.397235,...,0.237625,0.289363,0.358554,0.387324,0.342490,0.385267,0.358895,0.280707,0.406265,0.403278
1217,1218,P58371,536,0.402204,AADA,0.289796,0.384406,0.402204,0.352460,0.293018,...,0.250101,0.307452,0.305608,0.385235,0.388395,0.286248,0.366128,0.341487,0.325643,0.384271


In [11]:
result = merops.merge(ser[:500], left_on= 'UniProt_ID', right_on='accession')

In [14]:
result

,Unnamed: 0,UniProt_ID,MEROPS_ID,rank,accession,length,C,best_band,AAAA,AAAD,...,ADDA,ADDD,DAAA,DAAD,DADA,DADD,DDAA,DDAD,DDDA,DDDD
0,57,A0JNK3,S01.278,271,A0JNK3,458,0.566325,AAAA,0.566325,0.321813,...,0.381749,0.287993,0.364050,0.299794,0.445250,0.459770,0.298431,0.432277,0.287760,0.337588
1,60,A6YFB5,S01.277,420,A6YFB5,459,0.541937,DADA,0.322652,0.345516,...,0.319815,0.338619,0.392007,0.490859,0.541937,0.415563,0.324990,0.396982,0.393226,0.394477
2,63,B0BNB9,S01.278,227,B0BNB9,458,0.575178,AAAA,0.575178,0.334863,...,0.398271,0.319149,0.450126,0.308471,0.385500,0.423282,0.408107,0.300728,0.398485,0.372047
3,64,G1PU69,S01.278,108,G1PU69,456,0.599190,AAAA,0.599190,0.351351,...,0.288618,0.295815,0.338007,0.360918,0.500530,0.341167,0.418478,0.438373,0.301958,0.350686
4,65,D2HH54,S01.278,222,D2HH54,458,0.575848,AAAA,0.575848,0.394814,...,0.378107,0.293623,0.401447,0.321841,0.459120,0.421591,0.401015,0.376423,0.294902,0.337958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,42595,T0FCD4,S41.004,99,T0FCD4,456,0.601290,DAAD,0.312687,0.341475,...,0.541977,0.323250,0.363093,0.601290,0.381797,0.269612,0.417420,0.444038,0.447013,0.378415
496,42612,V5WCJ9,S41.004,485,V5WCJ9,495,0.534350,DAAD,0.429142,0.379933,...,0.330635,0.379411,0.492037,0.534350,0.377665,0.280251,0.451964,0.477707,0.335093,0.465288
497,42614,V6HXZ6,S41.004,411,V6HXZ6,462,0.542937,DDDA,0.303499,0.382325,...,0.502681,0.417698,0.367882,0.532954,0.378482,0.369694,0.440296,0.460174,0.542937,0.295448
498,42616,U5DKM5,S13.003,88,U5DKM5,464,0.606026,ADDD,0.385301,0.383740,...,0.373289,0.606026,0.401205,0.386639,0.420152,0.246044,0.316746,0.414640,0.391540,0.296659


In [23]:
mer = merops['MEROPS_ID']

In [24]:
import requests
import pandas as pd
import time
import re
from bs4 import BeautifulSoup


# ============================================================
# INPUT
# ============================================================

# Your existing list:
#
# mer = [
#     "S01.001",
#     "S01.002",
#     "M01.001",
#     ...
# ]


# ============================================================
# SETTINGS
# ============================================================

MEROPS_URL = "https://www.ebi.ac.uk/merops/cgi-bin/substrates"
UNIPROT_URL = "https://rest.uniprot.org/uniprotkb/{}.fasta"

WINDOW = 10
MEROPS_DELAY = 1.0


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 Protease-substrate research"
})


# ============================================================
# CACHE
# ============================================================

merops_cache = {}
sequence_cache = {}


# ============================================================
# GET MEROPS SUBSTRATE TABLE
# ============================================================

def get_merops_substrates(merops_id):

    if merops_id in merops_cache:
        return merops_cache[merops_id]

    try:

        response = session.get(
            MEROPS_URL,
            params={"id": merops_id},
            timeout=60
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        tables = soup.find_all("table")

        for table in tables:

            rows = table.find_all("tr")

            if not rows:
                continue

            headers = [
                cell.get_text(" ", strip=True)
                for cell in rows[0].find_all(
                    ["th", "td"]
                )
            ]

            # Find the substrate table
            substrate_column = None

            for i, header in enumerate(headers):

                if "substrate" in header.lower():

                    substrate_column = i
                    break

            if substrate_column is None:
                continue

            data = []

            for row in rows[1:]:

                cells = [
                    cell.get_text(" ", strip=True)
                    for cell in row.find_all(
                        ["td", "th"]
                    )
                ]

                if len(cells) <= substrate_column:
                    continue

                # Keep the complete row
                if len(cells) < len(headers):
                    cells += [""] * (
                        len(headers) - len(cells)
                    )

                cells = cells[:len(headers)]

                row_dict = dict(
                    zip(headers, cells)
                )

                data.append(row_dict)

            merops_cache[merops_id] = data

            return data

        merops_cache[merops_id] = []

        return []

    except Exception as e:

        print(
            f"MEROPS ERROR {merops_id}: {e}"
        )

        merops_cache[merops_id] = []

        return []


# ============================================================
# GENERIC COLUMN FINDER
# ============================================================

def get_column(row, possible_names):

    normalized = {}

    for key in row.keys():

        clean = re.sub(
            r"[^a-z0-9]",
            "",
            str(key).lower()
        )

        normalized[clean] = key

    for name in possible_names:

        clean = re.sub(
            r"[^a-z0-9]",
            "",
            name.lower()
        )

        if clean in normalized:

            value = row[normalized[clean]]

            if value is not None:
                return str(value).strip()

    return ""


# ============================================================
# GET SUBSTRATE UniProt ID
# ============================================================

def get_substrate_uniprot(row):

    value = get_column(
        row,
        [
            "UniProt",
            "Uniprot",
            "UniProt ID",
            "UniProt accession",
            "Accession"
        ]
    )

    # Some MEROPS entries may contain additional text.
    # Try to extract a standard UniProt accession.

    match = re.search(
        r"\b[A-NR-Z][0-9][A-Z0-9]{3}[0-9]\b",
        value
    )

    if match:
        return match.group(0)

    return value


# ============================================================
# GET SUBSTRATE SEQUENCE FROM UniProt
# ============================================================

def get_uniprot_sequence(uniprot_id):

    if not uniprot_id:
        return None

    if uniprot_id in sequence_cache:
        return sequence_cache[uniprot_id]

    try:

        # Remove isoform suffix if present
        # e.g. P12345-2 -> P12345
        base_id = uniprot_id.split("-")[0]

        url = UNIPROT_URL.format(
            base_id
        )

        response = session.get(
            url,
            timeout=60
        )

        if response.status_code != 200:

            sequence_cache[uniprot_id] = None

            return None

        sequence = "".join(
            line.strip()
            for line in response.text.splitlines()
            if not line.startswith(">")
        )

        sequence_cache[
            uniprot_id
        ] = sequence

        return sequence

    except Exception as e:

        print(
            f"UniProt ERROR {uniprot_id}: {e}"
        )

        sequence_cache[
            uniprot_id
        ] = None

        return None


# ============================================================
# PARSE CLEAVAGE POSITION
# ============================================================

def parse_cleavage_position(cleavage_site):

    if not cleavage_site:
        return None

    # Examples:
    #
    # Gly373+Ala
    # Arg195+Ser
    # Lys1754+Ser
    #
    # The number corresponds to P1.

    match = re.search(
        r"[A-Za-z]{1,3}[- ]?(\d+)\s*\+",
        cleavage_site
    )

    if match:

        return int(
            match.group(1)
        )

    return None


# ============================================================
# EXTRACT ±10 AA WINDOW
# ============================================================

def get_binding_window(
    sequence,
    cleavage_position,
    window=10
):

    if (
        not sequence
        or cleavage_position is None
    ):
        return None

    # MEROPS uses 1-based residue numbering.
    #
    # If cleavage is:
    #
    # residue 100 | residue 101
    #       P1    |    P1'
    #
    # Python cleavage boundary = index 100

    cut_index = cleavage_position

    left = max(
        0,
        cut_index - window
    )

    right = min(
        len(sequence),
        cut_index + window
    )

    region = sequence[
        left:right
    ]

    marker = cut_index - left

    return (
        region[:marker]
        + "|"
        + region[marker:]
    )


# ============================================================
# MAIN LOOP
# ============================================================

results = []

failed_merops = []

for i, merops_id in enumerate(
    mer,
    start=1
):

    print(
        f"[{i}/{len(mer)}] {merops_id}"
    )

    # --------------------------------------------------------
    # Get substrate records
    # --------------------------------------------------------

    substrates = get_merops_substrates(
        merops_id
    )

    print(
        f"    {len(substrates)} substrate records"
    )

    if not substrates:
        failed_merops.append(
            merops_id
        )

    # --------------------------------------------------------
    # Process every substrate
    # --------------------------------------------------------

    for substrate in substrates:

        # ----------------------------------------------------
        # Substrate information
        # ----------------------------------------------------

        substrate_name = get_column(
            substrate,
            [
                "Substrate",
                "Substrate name"
            ]
        )

        substrate_uniprot = (
            get_substrate_uniprot(
                substrate
            )
        )

        # ----------------------------------------------------
        # Cleavage site
        # ----------------------------------------------------

        cleavage_site = get_column(
            substrate,
            [
                "Cleavage Site",
                "Cleavage",
                "Site"
            ]
        )

        # ----------------------------------------------------
        # Get full substrate sequence
        # ----------------------------------------------------

        substrate_sequence = None

        if substrate_uniprot:

            substrate_sequence = (
                get_uniprot_sequence(
                    substrate_uniprot
                )
            )

        # ----------------------------------------------------
        # Synthetic substrate
        #
        # If there is no UniProt sequence, look for a
        # sequence directly in the MEROPS record.
        # ----------------------------------------------------

        if not substrate_sequence:

            substrate_sequence = get_column(
                substrate,
                [
                    "Sequence",
                    "Substrate sequence",
                    "Peptide sequence"
                ]
            )

            if not substrate_sequence:
                substrate_sequence = None

        # ----------------------------------------------------
        # Cleavage position
        # ----------------------------------------------------

        cleavage_position = (
            parse_cleavage_position(
                cleavage_site
            )
        )

        # ----------------------------------------------------
        # ±10 AA sequence
        # ----------------------------------------------------

        binding_window = (
            get_binding_window(
                substrate_sequence,
                cleavage_position,
                WINDOW
            )
        )

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        results.append({

            "merops_id":
                merops_id,

            "substrate":
                substrate_name,

            "substrate_uniprot":
                substrate_uniprot,

            "substrate_sequence":
                substrate_sequence,

            "cleavage_site":
                cleavage_site,

            "cleavage_position":
                cleavage_position,

            "binding_window_±10":
                binding_window
        })

    # --------------------------------------------------------
    # Delay before next MEROPS request
    # --------------------------------------------------------

    time.sleep(
        MEROPS_DELAY
    )


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    results
)


# ============================================================
# SAVE
# ============================================================

df.to_csv(
    "merops_substrates_with_window10.csv",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINISHED")
print("=" * 70)

print(
    "MEROPS IDs:",
    len(mer)
)

print(
    "Total substrate records:",
    len(df)
)

print(
    "Records with substrate sequence:",
    df["substrate_sequence"].notna().sum()
)

print(
    "Records with ±10 AA window:",
    df["binding_window_±10"].notna().sum()
)

print(
    "MEROPS IDs with no substrate records:",
    len(failed_merops)
)

print(
    "\nCSV:",
    "merops_substrates_with_window10.csv"
)

display(
    df.head(20)
)

[1/1220] S01.278
    235 substrate records
[2/1220] S01.277
    5 substrate records
[3/1220] S01.278
    235 substrate records
[4/1220] S01.278
    235 substrate records
[5/1220] S01.278
    235 substrate records
[6/1220] S01.278
    235 substrate records
[7/1220] S01.278
    235 substrate records
[8/1220] S41.002
    2 substrate records
[9/1220] S01.278
    235 substrate records
[10/1220] S01.216
    63 substrate records
[11/1220] S01.278
    235 substrate records
[12/1220] S01.278
    235 substrate records
[13/1220] S01.494
    39 substrate records
[14/1220] S01.278
    235 substrate records
[15/1220] S01.214
    12 substrate records
[16/1220] S01.278
    235 substrate records
[17/1220] S01.216
    63 substrate records
[18/1220] S01.247
    21 substrate records
[19/1220] S01.278
    235 substrate records
[20/1220] S41.007
    18 substrate records
[21/1220] S01.232
    9 substrate records
[22/1220] S01.202
    0 substrate records
[23/1220] S01.199
    7 substrate records
[24/1220] S08

,merops_id,substrate,substrate_uniprot,substrate_sequence,cleavage_site,cleavage_position,binding_window_±10
0,S01.278,3-hydroxyacyl-CoA dehydrogenase type-2,Q99714,MAAACRSVKGLVAVITGGASGLGLATAERLVGQGASAVLLDLPNSG...,peptide-Cys58+Val-peptide,58.0,AQAKKLGNNC|VFAPADVTSE
1,S01.278,"Actin, alpha cardiac muscle 1",P68032,MCDDEETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Val19+Lys-peptide,19.0,LVCDNGSGLV|KAGFAGDDAP
2,S01.278,"Actin, alpha cardiac muscle 1",P68032,MCDDEETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Leu107+Thr-peptide,107.0,VAPEEHPTLL|TEAPLNPKAN
3,S01.278,"Actin, alpha cardiac muscle 1",P68032,MCDDEETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Thr108+Glu-peptide,108.0,APEEHPTLLT|EAPLNPKANR
4,S01.278,"actin, alpha skeletal muscle",P68133,MCDEDETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Val19+Lys-peptide,19.0,LVCDNGSGLV|KAGFAGDDAP
5,S01.278,"actin, alpha skeletal muscle",P68133,MCDEDETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Leu107+Thr-peptide,107.0,VAPEEHPTLL|TEAPLNPKAN
6,S01.278,"actin, alpha skeletal muscle",P68133,MCDEDETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Val221+Ala-peptide,221.0,RDIKEKLCYV|ALDFENEMAT
7,S01.278,"actin, alpha skeletal muscle",P68133,MCDEDETTALVCDNGSGLVKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Thr108+Glu-peptide,108.0,APEEHPTLLT|EAPLNPKANR
8,S01.278,"Actin, aortic smooth muscle",P62736,MCEEEDSTALVCDNGSGLCKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Cys19+Lys-peptide,19.0,LVCDNGSGLC|KAGFAGDDAP
9,S01.278,"Actin, aortic smooth muscle",P62736,MCEEEDSTALVCDNGSGLCKAGFAGDDAPRAVFPSIVGRPRHQGVM...,peptide-Leu107+Thr-peptide,107.0,VAPEEHPTLL|TEAPLNPKAN


In [27]:
df.to_csv(r"C:\Users\apkar\OneDrive\Documents\iGEM\Head\Linker design\Serine_Protease_Substrates.csv")

In [25]:
df['binding_window_±10'].nunique()

3758